# DPMM Analysis of Efron-Morris Baseball Data

This notebook implements the full Gibbs sampler for the Dirichlet Process Mixture Model (DPMM) as described in MacEachern (1998) for the Efron-Morris (1975) baseball batting average data.


## MacEachern (1998) Model Specification

$$X_i \mid (\theta_i, z_i) \sim \text{Binomial}(z_i, \theta_i)$$
$$\theta_i \mid G \sim G$$
$$G \mid (M, \nu) \sim \text{DP}(M G_{0,\nu})$$
$$G_{0,\nu} \sim \text{Beta}(\alpha, \beta)$$
$$\frac{\alpha}{\alpha + \beta} \sim \text{Beta}(\nu_1, \nu_2)$$
$$\alpha + \beta = 216.6$$
$$M \sim \text{Lognormal}(2.81, 1.186^2)$$

where:
- $X_i$ = number of hits for player $i$ in first 45 at-bats
- $z_i$ = number of at-bats (45 for all players)
- $\theta_i$ = true batting ability for player $i$
- $G_{0,\nu}$ follows $\text{Beta}(\alpha, \beta)$ distribution 


## 1. Gibbs Probability Offsets
The conditional probabilities for assigning player $i$ to an existing or new cluster are derived from the **integrated Binomial-Beta marginal distribution**. For a cluster $j$ with sufficient statistics:

$$\alpha_j = \alpha + \sum_{l \neq i | s_l = j} X_l$$
$$\beta_j = \beta + \sum_{l \neq i | s_l = j} (z_l - X_l)$$

The marginal likelihood for assigning player $i$ (with $X_i$ hits in $z_i$ at-bats) to cluster $j$ is:

$$p(X_i \mid z_i, \text{cluster } j) \propto \frac{B(\alpha_j + X_i, \beta_j + z_i - X_i)}{B(\alpha_j, \beta_j)}$$

This follows from the Beta-Binomial conjugacy:
- Prior: $\theta_j \sim \text{Beta}(\alpha_j, \beta_j)$
- Likelihood: $X_i \mid \theta_j \sim \text{Binomial}(z_i, \theta_j)$
- Marginal: $\text{Beta-Binomial}(\alpha_j, \beta_j, z_i)$

### Gibbs Sampling Formulas
**Existing clusters** ($j = 1, \dots, k^-$):

$$q_j \propto \tilde{n}_j^- \frac{B(\alpha_j + X_i, \beta_j + z_i - X_i)}{B(\alpha_j, \beta_j)}$$

**New cluster** ($q_0$):

$$q_0 \propto \tilde{M} \frac{B(\alpha + X_i, \beta + z_i - X_i)}{B(\alpha, \beta)}$$

where:
- $\tilde{n}_j^-$ is the number of observations in cluster $j$ excluding $i$
- $\tilde{M}$ is the prior concentration parameter (derived from preintegration)


## 2. Exact $\nu_1, \nu_2$ Hyperparameters
The hyperparameters $\nu_1$ and $\nu_2$ control the prior distribution on the base measure mean:

$$\frac{\alpha}{\alpha + \beta} \sim \text{Beta}(\nu_1, \nu_2)$$

Following MacEachern (1998), these are determined via **moment matching** from the data:

**Mean matching**:

$$\frac{\nu_1}{\nu_1 + \nu_2} = 0.265$$

where $0.265$ is the overall batting average:
$$\frac{\sum_{i=1}^{18} X_i}{\sum_{i=1}^{18} z_i} = \frac{211}{810} = 0.265$$

**Variance matching**:

$$\nu_1 + \nu_2 = 811$$

This is derived from matching the sample standard deviation:

$$\text{Var}\left(\frac{\alpha}{\alpha+\beta}\right) = \frac{0.265 \times 0.735}{810}$$

Solving the system of equations yields the **exact hyperparameters**:

$$\boxed{\nu_1 = 214.915}$$

$$\boxed{\nu_2 = 596.085}$$

These values ensure the prior distribution on $\alpha/(\alpha+\beta)$ has:
| Property | Value |
|----------|-------|
| **Mean** | $0.265$ |
| **Variance** | $\frac{0.265 \times 0.735}{811}$ |
| **Sum** | $811$ |



## 3. Updating $M$ based on Posterior Density of $M$
We update $M$ by sampling from its posterior distribution:

$$p(M \mid \boldsymbol{s}) \propto p(\boldsymbol{s} \mid M) \, p(M)$$

where the prior is
$$M \sim \text{Lognormal}(2.81, 1.186^2)$$

and the likelihood (from the Chinese Restaurant Process) is
$$p(\boldsymbol{s} \mid M) = \frac{M^k \, \Gamma(M)}{\Gamma(M + n)} \prod_{j=1}^k (n_j - 1)!$$

Taking the logarithm gives the full conditional posterior:
$$\log p(M \mid \boldsymbol{s}) = k\log M + \log\Gamma(M) - \log\Gamma(M+n) + \log \text{Lognormal}(M \mid 2.81, 1.186^2)$$

This posterior combines prior beliefs with information from the current cluster configuration $\boldsymbol{s}$, ensuring $M$ is updated based on the observed data structure,


## 4. Update $\alpha$ based on Posterior Density of $\alpha$
The parameter $\alpha$ (first shape parameter of the base Beta distribution) is updated from its posterior distribution using Metropolis-Hastings. For $k$ clusters with means $\theta_1^*, \dots, \theta_k^*$, the posterior $\alpha$ is proportional to:

$$\boxed{p(\alpha \mid \boldsymbol{\theta}^*) \propto B(\alpha, c-\alpha)^{-k} \prod_{j=1}^k \theta_j^{*\alpha-1} (1-\theta_j^*)^{c-\alpha-1} \cdot \alpha^{\nu_1-1}(c-\alpha)^{\nu_2-1}}$$

where:
- $c=\alpha + \beta = 216.6$, which is fixed 
- k: Number of clusters 
- $\boldsymbol{\theta}^* = (\theta_1^*, \dots, \theta_k^*)$, denoting the cluster means from 1 to k 
-  $\nu_1, \nu_2$ is Hyperparameters (see Section 2)

### Derivation
The full conditional posterior density is derived by combining:
-  **Likelihood**: The cluster means $\theta_j^*$ are i.i.d. from $\text{Beta}(\alpha, c-\alpha)$, so likelihoood is the product of $p(\theta_j^* \mid  \alpha)$ where $j=1, \dots, k$, 
   $$p(\theta^* \mid \alpha) \propto \prod_{j=1}^k \frac{(\theta_j^*)^{\alpha-1}(1-\theta_j^*)^{c-\alpha-1}}{B(\alpha, c-\alpha)}$$
- **Prior**: $p(\alpha)\propto \alpha^{\nu_1-1}(c-\alpha)^{\nu_2-1}$ from the Beta prior on $\alpha/(\alpha+\beta)$

Multiplying and dropping constants yields the expression above. This density is used in the **Metropolis-Hastings step** for sampling $\alpha$.


## 5. Predictive Estimation
The goal is to predict each player's true batting ability $\theta_i$ for the remainder of the season. We compute the **posterior predictive expectation**:

$$E[\theta_i \mid \text{data}]$$

This is the Bayesian optimal point estimate under squared error loss.

### Computation
For each MCMC sample $t = 1, \dots, T$:
1). Identify cluster assignment $s_i^{(t)}$ for player $i$
2). Retrieve the corresponding cluster mean $\theta_{s_i^{(t)}}^{* (t)}$
3). This is the posterior draw of $\theta_i$ given the current cluster configuration

The predictive mean is then:

$$\boxed{\hat{\theta}_i = \frac{1}{T} \sum_{t=1}^T \theta_{s_i^{(t)}}^{* (t)}}$$

### Interpretation
This provides:
- **Point estimate**: $\hat{\theta}_i$ (mean)
- **Uncertainty quantification**: Posterior variance and credible intervals from the MCMC samples
- **Proper shrinkage**: Players with similar performance are pooled together via clustering

For each player $i$:
- If $s_i^{(t)} = j$, then $\theta_i^{(t)} = \theta_j^{*(t)}$
- Average over all $t$ to get $E[\theta_i \mid \text{data}]$


## Running Python code 

In [1]:
import numpy as np
import pandas as pd
from scipy.special import betaln
import matplotlib.pyplot as plt
import math

# ============================================================================
# DATA: Efron-Morris (1975) / MacEachern (1998) Baseball Data
# ============================================================================

def load_baseball_data():
    players = ["Clemente", "F.Robinson", "F.Howard", "Johnstone", "Berry", "Spencer", 
               "Kessinger", "Alvarado", "Santo", "Swoboda", "Unser", "Williams", 
               "Scott", "Petrocelli",  "E.Rodriguez", "Campaneris", "Munson", "Alvis"]

    # First 45 at-bats
    hits = np.array([
        18, 17, 16, 15, 14, 14, 13, 12, 11,
        11, 10, 10, 10, 10, 10, 9, 8, 7
    ])
    
    at_bats = np.full(18, 45)

    # Remaining at-bats in the 1970 season
    remainder_hits = np.array([
        127, 127, 144, 61, 114, 126, 154, 29, 137,
        46, 73, 69, 132, 142, 42, 159, 129, 14
    ])

    remainder_at_bats = np.array([
        367, 426, 521, 275, 418, 466, 586, 138, 510,
        200, 277, 270, 435, 538, 186, 558, 408, 70
    ])

    return {
        "players": players,  
        "hits": hits,  
        "at_bats": at_bats,
        "averages": hits / at_bats,   
        "remainder_hits": remainder_hits,
        "remainder_at_bats": remainder_at_bats,
        "remainder_avg": remainder_hits / remainder_at_bats,
        "n_players": len(players)
    }

# ============================================================================
# MacEachern (1998) MDP Model Implementation
# ============================================================================

class DPMM_MacEachern:
    def __init__(self, data: dict, n_iterations: int = 10000, n_burnin: int = 1000, seed: int = 42):
        np.random.seed(seed)
        self.data = data
        self.n = data['n_players']
        self.X = data['hits']
        self.z = data['at_bats']
        
        # Hyperparameters (MacEachern 1998, Section 2)
        self.c = 216.6
        # nu1 / (nu1 + nu2) = 0.265 and nu1 + nu2 = 811
        self.nu1 = 0.265 * 811.0
        self.nu2 = 811.0 - self.nu1
        self.M_mu = 2.81
        self.M_sigma = 1.186
        
        # MCMC parameters
        self.n_iterations = n_iterations
        self.n_burnin = n_burnin
        
        # Initialize state (Section 2)
        self.s = np.arange(self.n)  # s = (0, 1, ..., n-1)
        self.theta_star = self.X / self.z
        self.alpha = 0.265 * self.c
        self.M = np.random.lognormal(self.M_mu, self.M_sigma)
        
        # Trace storage
        self.samples = {'alpha': [], 'M': [], 'k': [], 'theta_i': []}

    def _log_beta_ratio(self, a_new, b_new, a_old, b_old):
        return betaln(a_new, b_new) - betaln(a_old, b_old)

    def update_cluster_assignments(self):
        """Step 1: Collapsed Gibbs sampler for s_i"""
        beta_param = self.c - self.alpha
        
        for i in range(self.n):
            # Temporarily exclude i
            s_i = self.s[i]
            self.s[i] = -1
            
            unique_clusters = [c for c in np.unique(self.s) if c != -1]
            k_minus = len(unique_clusters)
            
            log_q = np.zeros(k_minus + 1)
            
            # Existing clusters (q_j: 0, 1.. k-1)
            for idx, j in enumerate(unique_clusters):
                mask = (self.s == j)
                n_j = np.sum(mask)
                sum_X = np.sum(self.X[mask])
                sum_z = np.sum(self.z[mask])
                
                alpha_j = self.alpha + sum_X
                beta_j = beta_param + (sum_z - sum_X)
                
                # Formula: q_j proportional to n_j^- * B(alpha_j + X_i, beta_j + z_i - X_i) / B(alpha_j, beta_j)
                log_q[idx] = np.log(n_j) + self._log_beta_ratio(
                    alpha_j + self.X[i], beta_j + self.z[i] - self.X[i],
                    alpha_j, beta_j
                )
            
            # New cluster (q_0: k)
            log_q[k_minus] = np.log(self.M) + self._log_beta_ratio(
                self.alpha + self.X[i], beta_param + self.z[i] - self.X[i],
                self.alpha, beta_param
            )
            
            # Sample new cluster
            probs = np.exp(log_q - np.max(log_q))
            probs /= np.sum(probs)
            chosen_idx = np.random.choice(k_minus + 1, p=probs)
            
            if chosen_idx < k_minus:
                self.s[i] = unique_clusters[chosen_idx]
            else:
                # Assign to new unique cluster index
                new_cluster_id = max(self.s) + 1 if np.max(self.s) >= 0 else 0
                self.s[i] = new_cluster_id

        # Relabel cluster indices to 0, 1, ..., k-1 
        _, self.s = np.unique(self.s, return_inverse=True)

    def update_cluster_means(self):
        """Step 2: Sample cluster parameters theta_j*"""
        k = len(np.unique(self.s))
        self.theta_star = np.zeros(k)
        beta_param = self.c - self.alpha
        
        for j in range(k):
            mask = (self.s == j)
            sum_X = np.sum(self.X[mask])
            sum_z = np.sum(self.z[mask])
            
            a_post = self.alpha + sum_X
            b_post = beta_param + (sum_z - sum_X)
            self.theta_star[j] = np.random.beta(a_post, b_post)

    def update_alpha(self, prop_sd=0.08):
        """Step 3: Update alpha via Metropolis-Hastings using paper density.
        Note: With c=216.6 and n=18, prop_sd=0.08 is a fine-tuned standard deviation
               yielding an MH acceptance rate of approximately 20–40%.

        Theoretical basis: Optimal RW-MH rule: prop_sd = (2.38 / sqrt(d)) * sigma_post
             where d = 1 for a 1D target distribution (Gelman et al., 1996).
        """
        def log_post_alpha(a):
            if a <= 0 or a >= self.c:
                return -np.inf
            b = self.c - a
            k = len(self.theta_star)
            
            # Full conditional from MacEachern (1998)
            log_lik = -k * betaln(a, b) + (a - 1) * np.sum(np.log(self.theta_star)) + (b - 1) * np.sum(np.log(1 - self.theta_star))
            log_prior = (self.nu1 - 1) * np.log(a) + (self.nu2 - 1) * np.log(b)
            return log_lik + log_prior

        # Random walk Metropolis proposal
        proposed_alpha = np.random.normal(self.alpha, prop_sd)
        if 0 < proposed_alpha < self.c:
            log_acc = log_post_alpha(proposed_alpha) - log_post_alpha(self.alpha)
            if np.log(np.random.uniform()) < log_acc:
                self.alpha = proposed_alpha

    def update_M(self, prop_sd=0.15):
        """Step 4: Update concentration parameter M"""
        def log_post_M(m):
            if m <= 0:
                return -np.inf
            k = len(np.unique(self.s))
            # CRP marginal log-likelihood for M + Lognormal prior
            log_lik = k * np.log(m) + math.lgamma(m) - math.lgamma(m + self.n)
            log_prior = -0.5 * ((np.log(m) - self.M_mu) / self.M_sigma) ** 2 - np.log(m)
            return log_lik + log_prior

        proposed_M = np.exp(np.random.normal(np.log(self.M), prop_sd))
        log_acc = log_post_M(proposed_M) - log_post_M(self.M)
        if np.log(np.random.uniform()) < log_acc:
            self.M = proposed_M

    def run_mcmc(self):
        print("Running MCMC Sampling...")
        for iteration in range(self.n_iterations):
            self.update_cluster_assignments()
            self.update_cluster_means()
            self.update_alpha()
            self.update_M()
            
            if iteration >= self.n_burnin:
                self.samples['alpha'].append(self.alpha)
                self.samples['M'].append(self.M)
                self.samples['k'].append(len(np.unique(self.s)))
                # Individual player parameters theta_i
                self.samples['theta_i'].append(self.theta_star[self.s])

        # Convert traces to arrays
        for k in self.samples:
            self.samples[k] = np.array(self.samples[k])
        print("MCMC Complete.")

    def print_results(self):
        mdp_estimates = np.mean(self.samples['theta_i'], axis=0)
        
        df = pd.DataFrame({
            'Player': self.data['players'],
            'X/z': np.round(self.data['averages'], 3),
            'Rest of Season': np.round(self.data['remainder_avg'], 3),
            'MDP Est': np.round(mdp_estimates, 3)
        })
        
        print("\n" + "=" * 45)
        print(" Table 1 Comparison: MacEachern (1998) MDP ")
        print("=" * 45)
        print(df.to_string(index=False))
        
        print("\nHyperparameter Posterior Means:")
        print(f"  alpha : {np.mean(self.samples['alpha']):.2f}")
        print(f"  beta  : {self.c - np.mean(self.samples['alpha']):.2f}")
        print(f"  M     : {np.mean(self.samples['M']):.2f}")
        print(f"  k     : {np.mean(self.samples['k']):.2f}")


# ============================================================================
# Main Execution
# ============================================================================
#if __name__ == "__main__":
#    data = load_baseball_data()
#    model = DPMM_MacEachern(data=data, n_iterations=15000, n_burnin=3000)
#    model.run_mcmc()
#    model.print_results()

In [2]:
# ============================================================================
# Main Execution (Direct)
# ============================================================================
data = load_baseball_data()
mod_dpmm = DPMM_MacEachern(data=data, n_iterations=15000, n_burnin=3000)
mod_dpmm.run_mcmc()
mod_dpmm.print_results()

Running MCMC Sampling...
MCMC Complete.

 Table 1 Comparison: MacEachern (1998) MDP 
     Player   X/z  Rest of Season  MDP Est
   Clemente 0.400           0.346    0.275
 F.Robinson 0.378           0.298    0.273
   F.Howard 0.356           0.276    0.270
  Johnstone 0.333           0.222    0.268
      Berry 0.311           0.273    0.265
    Spencer 0.311           0.270    0.265
  Kessinger 0.289           0.263    0.262
   Alvarado 0.267           0.210    0.259
      Santo 0.244           0.269    0.257
    Swoboda 0.244           0.230    0.257
      Unser 0.222           0.264    0.254
   Williams 0.222           0.256    0.254
      Scott 0.222           0.303    0.254
 Petrocelli 0.222           0.264    0.254
E.Rodriguez 0.222           0.226    0.254
 Campaneris 0.200           0.285    0.251
     Munson 0.178           0.316    0.248
      Alvis 0.156           0.200    0.245

Hyperparameter Posterior Means:
  alpha : 54.89
  beta  : 161.71
  M     : 7.86
  k     : 7.45



## References
MacEachern, S. N. (1998). Computational methods for mixture of Dirichlet process models. In D. Dey, P. Müller, & D. Sinha (Eds.), Practical nonparametric and semiparametric Bayesian statistics (pp. 23–43). Springer.

Efron, B., & Morris, C. (1975). Data analysis using Stein's estimator and its generalizations. Journal of the American Statistical Association, 70(350), 311–319.